# Imports

In [ ]:
import os
os.environ["XLA_FLAGS"] = (
    "--xla_force_host_platform_device_count=8"  # Use 8 CPU cores for JAX pmap
)

from mechanicalmetamaterialcloaks.geometry import KagomeGeometry, CloakKagomeGeometry
from mechanicalmetamaterialcloaks.plotting import (
    generate_animation,
    plot_geometry,
    generate_several_animations,
    plot_geometry_field_overlaid,
)
from problems.kagome_static_cloaking import (
    ForwardInput,
    ForwardProblem,
    OptimizationProblem,
)

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import jax
import jax.numpy as jnp
from pathlib import Path
from mechanicalmetamaterialcloaks.utils import save_data, load_data, SolutionData
from typing import Optional

jax.config.update("jax_enable_x64", True)  # enable float64 type

plt.style.use(["science", "grid"])
%matplotlib widget

# Plotting functions

In [ ]:
def plot_objective_iterations(
    optimization: OptimizationProblem, optimization_filename: Optional[str] = None
):
    fig, axes = plt.subplots(
        nrows=3, figsize=(10, 7), sharex=True, constrained_layout=True
    )
    axes[0].set(ylabel="Objective")
    axes[0].plot(optimization.objective_values, lw=3, color="#2980b9")
    axes[1].set(ylabel="Angle constraints violation")
    axes[1].plot(optimization.constraints_violation["angles"], lw=3, color="#c0392b")
    axes[1].axhline(y=0, color="black")
    axes[2].set(ylabel="Edge length constraints violation")
    axes[2].plot(
        optimization.constraints_violation["edge_lengths"], lw=3, color="#c0392b"
    )
    axes[2].axhline(y=0, color="black")
    axes[-1].set(xlabel="Iteration")

    if optimization_filename is not None:
        path = Path(
            f"../out/{optimization.name}/{optimization_filename}/objective_iterations.png"
        )
        path.parent.mkdir(
            parents=True, exist_ok=True
        )  # Make sure parents directories exist
        fig.savefig(str(path), dpi=300)


# DELTA DIFFERENCE FUNCTIONS
def difference_displacement_field_centroids(
    solutionData_cg: SolutionData,
    solutionData_mg: SolutionData,
    cloaked_geometry: CloakKagomeGeometry,
):
    dimension_less = jnp.array(
        [1 / cloaked_geometry.spacing, 1 / cloaked_geometry.spacing, 1]
    )
    return (
        (
            (
                (
                    solutionData_cg.fields[
                        -1,
                        0,
                        cloaked_geometry.cgIDs_surronding_area,
                        :,
                    ]
                    - solutionData_mg.fields[
                        -1,
                        0,
                        cloaked_geometry.mgIDs_surronding_area,
                        :,
                    ]
                )
                * dimension_less
            )
            ** 2
        ).sum()
        ** 0.5
    ) / (
        (
            solutionData_mg.fields[-1, 0, cloaked_geometry.mgIDs_surronding_area, :]
            * dimension_less
        )
        ** 2
    ).sum() ** 0.5


def difference_ux_centroids(
    solutionData_cg: SolutionData,
    solutionData_mg: SolutionData,
    cloaked_geometry: CloakKagomeGeometry,
):
    return (
        (
            (
                solutionData_cg.fields[
                    -1,
                    0,
                    cloaked_geometry.cgIDs_surronding_area,
                    0,
                ]
                - solutionData_mg.fields[
                    -1, 0, cloaked_geometry.mgIDs_surronding_area, 0
                ]
            )
            ** 2
        ).sum()
        ** 0.5
    ) / (
        (solutionData_mg.fields[-1, 0, cloaked_geometry.mgIDs_surronding_area, 0]) ** 2
    ).sum() ** 0.5


def difference_uy_centroids(
    solutionData_cg: SolutionData,
    solutionData_mg: SolutionData,
    cloaked_geometry: CloakKagomeGeometry,
):
    return (
        (
            (
                solutionData_cg.fields[
                    -1,
                    0,
                    cloaked_geometry.cgIDs_surronding_area,
                    1,
                ]
                - solutionData_mg.fields[
                    -1, 0, cloaked_geometry.mgIDs_surronding_area, 1
                ]
            )
            ** 2
        ).sum()
        ** 0.5
    ) / (
        (solutionData_mg.fields[-1, 0, cloaked_geometry.mgIDs_surronding_area, 1]) ** 2
    ).sum() ** 0.5


def difference_theta_centroids(
    solutionData_cg: SolutionData,
    solutionData_mg: SolutionData,
    cloaked_geometry: CloakKagomeGeometry,
):
    return (
        (
            (
                solutionData_cg.fields[
                    -1,
                    0,
                    cloaked_geometry.cgIDs_surronding_area,
                    2,
                ]
                - solutionData_mg.fields[
                    -1, 0, cloaked_geometry.mgIDs_surronding_area, 2
                ]
            )
            ** 2
        ).sum()
        ** 0.5
    ) / (
        (solutionData_mg.fields[-1, 0, cloaked_geometry.mgIDs_surronding_area, 2]) ** 2
    ).sum() ** 0.5


def difference_ux_centroids_dimension_less(
    solutionData_cg: SolutionData,
    solutionData_mg: SolutionData,
    cloaked_geometry: CloakKagomeGeometry,
    optimization: OptimizationProblem,
):
    return (
        (
            (
                (
                    solutionData_cg.fields[
                        -1,
                        0,
                        cloaked_geometry.cgIDs_surronding_area,
                        0,
                    ]
                    - solutionData_mg.fields[
                        -1,
                        0,
                        cloaked_geometry.mgIDs_surronding_area,
                        0,
                    ]
                )
                / cloaked_geometry.spacing
            )
            ** 2
        ).sum()
    ) / (
        (
            solutionData_mg.fields[-1, 0, cloaked_geometry.mgIDs_surronding_area, :]
            * optimization.dimension_less
        )
        ** 2
    ).sum()


def difference_uy_centroids_dimension_less(
    solutionData_cg: SolutionData,
    solutionData_mg: SolutionData,
    cloaked_geometry: CloakKagomeGeometry,
    optimization: OptimizationProblem,
):
    return (
        (
            (
                (
                    solutionData_cg.fields[
                        -1,
                        0,
                        cloaked_geometry.cgIDs_surronding_area,
                        1,
                    ]
                    - solutionData_mg.fields[
                        -1,
                        0,
                        cloaked_geometry.mgIDs_surronding_area,
                        1,
                    ]
                )
                / cloaked_geometry.spacing
            )
            ** 2
        ).sum()
    ) / (
        (
            solutionData_mg.fields[-1, 0, cloaked_geometry.mgIDs_surronding_area, :]
            * optimization.dimension_less
        )
        ** 2
    ).sum()


def difference_utheta_centroids_dimension_less(
    solutionData_cg: SolutionData,
    solutionData_mg: SolutionData,
    cloaked_geometry: CloakKagomeGeometry,
    optimization: OptimizationProblem,
):
    return (
        (
            (
                solutionData_cg.fields[
                    -1,
                    0,
                    cloaked_geometry.cgIDs_surronding_area,
                    2,
                ]
                - solutionData_mg.fields[
                    -1, 0, cloaked_geometry.mgIDs_surronding_area, 2
                ]
            )
            ** 2
        ).sum()
    ) / (
        (
            solutionData_mg.fields[-1, 0, cloaked_geometry.mgIDs_surronding_area, :]
            * optimization.dimension_less
        )
        ** 2
    ).sum()


# GENERATE DELTA DIFFERENCE EVOLUTION
def generate_delta_difference(
    solutionData_cg: SolutionData,
    solutionData_mg: SolutionData,
    cloaked_geometry: CloakKagomeGeometry,
    title: str,
):
    _initial_delta = difference_displacement_field_centroids(
        solutionData_cg, solutionData_mg, cloaked_geometry
    )
    # plot
    fig, ax = plt.subplots()
    geometry = ["initial guess"]
    delta_value = [_initial_delta]
    bar_labels = ["red"]
    bar_colors = ["tab:red"]
    ax.bar(geometry, delta_value, label=bar_labels, color=bar_colors)
    ax.set_ylabel("Delta value")
    ax.set_title(title)
    plt.show()


def generate_delta_difference_optimized_and_initial_geometry_and_save(
    initial_solutionData_cg: SolutionData,
    solutionData_mg: SolutionData,
    optimized_solutionData_cg: SolutionData,
    cloaked_geometry: CloakKagomeGeometry,
    title: str,
    out_filename: str,
):
    fig, ax = plt.subplots()
    _initial_delta = difference_displacement_field_centroids(
        initial_solutionData_cg, solutionData_mg, cloaked_geometry
    )
    _optimized_delta = difference_displacement_field_centroids(
        optimized_solutionData_cg, solutionData_mg, cloaked_geometry
    )

    # plot
    geometry = ["initial guess", "optimized response"]
    delta_value = [_initial_delta, _optimized_delta]
    bar_labels = ["red", "blue"]
    bar_colors = ["tab:red", "tab:blue"]
    ax.bar(geometry, delta_value, label=bar_labels, color=bar_colors)
    ax.set_ylabel("Delta value")
    ax.set_title(title)
    plt.savefig(out_filename + ".jpg", bbox_inches="tight", dpi=150)


def generate_delta_uk_difference_optimized_and_initial_geometry(
    uk: str,
    initial_solutionData_cg: SolutionData,
    solutionData_mg: SolutionData,
    optimized_solutionData_cg: SolutionData,
    cloaked_geometry: CloakKagomeGeometry,
):
    if uk == "ux":
        _initial_delta = difference_ux_centroids(
            initial_solutionData_cg, solutionData_mg, cloaked_geometry
        )
        _optimized_delta = difference_ux_centroids(
            optimized_solutionData_cg, solutionData_mg, cloaked_geometry
        )
        title = "ux delta difference - opimized geometry and initial geometry"

    if uk == "uy":
        _initial_delta = difference_uy_centroids(
            initial_solutionData_cg, solutionData_mg, cloaked_geometry
        )
        _optimized_delta = difference_uy_centroids(
            optimized_solutionData_cg, solutionData_mg, cloaked_geometry
        )
        title = "uy delta difference - opimized geometry and initial geometry"

    if uk == "theta":
        _initial_delta = difference_theta_centroids(
            initial_solutionData_cg, solutionData_mg, cloaked_geometry
        )
        _optimized_delta = difference_theta_centroids(
            optimized_solutionData_cg, solutionData_mg, cloaked_geometry
        )
        title = "theta delta difference - opimized geometry and initial geometry"
    if uk == "ux" or uk == "uy" or uk == "theta":
        fig, ax = plt.subplots()
        ax.plot(
            initial_solutionData_cg.timepoints,
            _initial_delta,
            c="blue",
            label="initial geometry",
        )
        ax.plot(
            optimized_solutionData_cg.timepoints,
            _optimized_delta,
            c="red",
            label="optimized geometry",
        )
        ax.set(ylabel="difference (dimension less)", xlabel="time (s)", title=title)
        ax.legend()
        plt.show()


def generate_delta_difference_square_uks_contribution(
    solutionData_cg: SolutionData,
    solutionData_mg: SolutionData,
    cloaked_geometry: CloakKagomeGeometry,
    optimization: OptimizationProblem,
    suffix_title: str = "",
):
    contribution_ux_delta_square = difference_ux_centroids_dimension_less(
        solutionData_cg, solutionData_mg, cloaked_geometry, optimization
    )
    contribution_uy_delta_square = difference_uy_centroids_dimension_less(
        solutionData_cg, solutionData_mg, cloaked_geometry, optimization
    )
    contribution_utheta_delta_square = difference_utheta_centroids_dimension_less(
        solutionData_cg, solutionData_mg, cloaked_geometry, optimization
    )
    fig, ax = plt.subplots()
    ax.plot(
        solutionData_cg.timepoints, contribution_ux_delta_square, c="blue", label="ux"
    )
    ax.plot(
        solutionData_cg.timepoints, contribution_uy_delta_square, c="red", label="uy"
    )
    ax.plot(
        solutionData_cg.timepoints,
        contribution_utheta_delta_square,
        c="green",
        label="utheta",
    )
    ax.set(
        ylabel="difference squared (dimension less)",
        xlabel="time (s)",
        title="Contribution of ux, uy, utheta in delta squared" + suffix_title,
    )
    ax.legend()
    plt.show()


# PLOT THE DISPLACEMENT OF TARGETED CENTROIDS IN THE MAP


def plot_displacement_1centroid_for_the_three_different_geometries(
    id_centroid_in_cg: int,
    id_centroids_in_mg: int,
    solution_data_cg_optimized_geometry: SolutionData,
    solution_data_cg_initial_guess: SolutionData,
    solution_data_mg: SolutionData,
    title: str = "",
    t_min: int = 0,
    t_max: int = None,
):
    if t_max == None:
        t_max = solution_data_cg_optimized_geometry.timepoints.shape[0]
    fig, axes = plt.subplots(
        nrows=3, figsize=(10, 10), sharex=True, constrained_layout=True
    )
    axes[0].set(ylabel="u_x displacement")
    axes[0].plot(
        solution_data_cg_optimized_geometry.timepoints[t_min:t_max],
        solution_data_cg_optimized_geometry.fields[
            t_min:t_max, 0, id_centroid_in_cg, 0
        ],
        lw=3,
        color="red",
        label="optimized geometry",
    )
    axes[0].plot(
        solution_data_cg_initial_guess.timepoints[t_min:t_max],
        solution_data_cg_initial_guess.fields[t_min:t_max, 0, id_centroid_in_cg, 0],
        lw=3,
        color="blue",
        label="initial guess geometry",
    )
    axes[0].plot(
        solution_data_mg.timepoints[t_min:t_max],
        solution_data_mg.fields[t_min:t_max, 0, id_centroids_in_mg, 0],
        lw=3,
        color="green",
        label="mother geometry",
    )
    axes[0].legend()

    axes[1].set(ylabel="u_y displacement")
    axes[1].plot(
        solution_data_cg_optimized_geometry.timepoints[t_min:t_max],
        solution_data_cg_optimized_geometry.fields[
            t_min:t_max, 0, id_centroid_in_cg, 1
        ],
        lw=3,
        color="red",
        label="optimized geometry",
    )
    axes[1].plot(
        solution_data_cg_initial_guess.timepoints[t_min:t_max],
        solution_data_cg_initial_guess.fields[t_min:t_max, 0, id_centroid_in_cg, 1],
        lw=3,
        color="blue",
        label="initial guess geometry",
    )
    axes[1].plot(
        solution_data_mg.timepoints[t_min:t_max],
        solution_data_mg.fields[t_min:t_max, 0, id_centroids_in_mg, 1],
        lw=3,
        color="green",
        label="mother geometry",
    )
    axes[1].legend()

    axes[2].set(ylabel="theta")
    axes[2].plot(
        solution_data_cg_optimized_geometry.timepoints[t_min:t_max],
        solution_data_cg_optimized_geometry.fields[
            t_min:t_max, 0, id_centroid_in_cg, 2
        ],
        lw=3,
        color="red",
        label="optimized geometry",
    )
    axes[2].plot(
        solution_data_cg_initial_guess.timepoints[t_min:t_max],
        solution_data_cg_initial_guess.fields[t_min:t_max, 0, id_centroid_in_cg, 2],
        lw=3,
        color="blue",
        label="initial guess geometry",
    )
    axes[2].plot(
        solution_data_mg.timepoints[t_min:t_max],
        solution_data_mg.fields[t_min:t_max, 0, id_centroids_in_mg, 2],
        lw=3,
        color="green",
        label="mother geometry",
    )
    axes[2].legend()

    axes[0].set_title(title)
    axes[-1].set(xlabel="times (s)")


def plot_displacement_1centroid(id_centroid: int, solution_data: SolutionData):
    fig, axes = plt.subplots(
        nrows=3, figsize=(10, 10), sharex=True, constrained_layout=True
    )
    axes[0].set(ylabel="u_x displacement")
    axes[0].plot(
        solution_data.timepoints,
        solution_data.fields[:, 0, id_centroid, 0],
        lw=3,
        color="red",
    )

    axes[1].set(ylabel="u_y displacement")
    axes[1].plot(
        solution_data.timepoints,
        solution_data.fields[:, 0, id_centroid, 1],
        lw=3,
        color="red",
    )

    axes[2].set(ylabel="theta")
    axes[2].plot(
        solution_data.timepoints,
        solution_data.fields[:, 0, id_centroid, 2],
        lw=3,
        color="red",
    )

    axes[-1].set(xlabel="times (s)")


def plot_displacement_1centroid_for_mg_and_initial_guess_cg(
    id_centroid_in_cg: int,
    id_centroids_in_mg: int,
    solution_data_cg_initial_guess: SolutionData,
    solution_data_mg: SolutionData,
    title: str = "",
    t_min: int = 0,
    t_max: int = None,
):
    if t_max == None:
        t_max = solution_data_cg_initial_guess.timepoints.shape[0]
    fig, axes = plt.subplots(
        nrows=3, figsize=(10, 10), sharex=True, constrained_layout=True
    )
    axes[0].set(ylabel="u_x displacement")
    axes[0].plot(
        solution_data_cg_initial_guess.timepoints[t_min:t_max],
        solution_data_cg_initial_guess.fields[t_min:t_max, 0, id_centroid_in_cg, 0],
        lw=3,
        color="blue",
        label="geometry with initial guess",
    )
    axes[0].plot(
        solution_data_mg.timepoints[t_min:t_max],
        solution_data_mg.fields[t_min:t_max, 0, id_centroids_in_mg, 0],
        lw=3,
        color="green",
    )
    axes[0].legend()

    axes[1].set(ylabel="u_y displacement")
    axes[1].plot(
        solution_data_cg_initial_guess.timepoints[t_min:t_max],
        solution_data_cg_initial_guess.fields[t_min:t_max, 0, id_centroid_in_cg, 1],
        lw=3,
        color="blue",
        label="geometry initial guess",
    )
    axes[1].plot(
        solution_data_mg.timepoints[t_min:t_max],
        solution_data_mg.fields[t_min:t_max, 0, id_centroids_in_mg, 1],
        lw=3,
        color="green",
    )
    axes[1].legend()

    axes[2].set(ylabel="theta")
    axes[2].plot(
        solution_data_cg_initial_guess.timepoints[t_min:t_max],
        solution_data_cg_initial_guess.fields[t_min:t_max, 0, id_centroid_in_cg, 2],
        lw=3,
        color="blue",
        label="geometry initial guess",
    )
    axes[2].plot(
        solution_data_mg.timepoints[t_min:t_max],
        solution_data_mg.fields[t_min:t_max, 0, id_centroids_in_mg, 2],
        lw=3,
        color="green",
    )
    axes[2].legend()

    axes[0].set_title(title)
    axes[-1].set(xlabel="times (s)")


def plot_displacement_several_centroids_for_mg(
    ids_centroids_in_mg: list,
    solution_data_mg: SolutionData,
    title: str = "",
    t_min: int = 0,
    t_max: int = None,
):
    if t_max == None:
        t_max = solution_data_mg.timepoints.shape[0]
    fig, axes = plt.subplots(
        nrows=3, figsize=(10, 10), sharex=True, constrained_layout=True
    )
    axes[0].set(ylabel="u_x displacement")
    for k in range(len(ids_centroids_in_mg)):
        axes[0].plot(
            solution_data_mg.timepoints[t_min:t_max],
            solution_data_mg.fields[t_min:t_max, 0, ids_centroids_in_mg[k], 0],
            lw=3,
            label=f"column {k}",
        )
    axes[0].legend()

    axes[1].set(ylabel="u_y displacement")
    for k in range(len(ids_centroids_in_mg)):
        axes[1].plot(
            solution_data_mg.timepoints[t_min:t_max],
            solution_data_mg.fields[t_min:t_max, 0, ids_centroids_in_mg[k], 1],
            lw=3,
            label=f"column {k}",
        )
    # axes[1].legend()

    axes[2].set(ylabel="theta")
    for k in range(len(ids_centroids_in_mg)):
        axes[2].plot(
            solution_data_mg.timepoints[t_min:t_max],
            solution_data_mg.fields[t_min:t_max, 0, ids_centroids_in_mg[k], 2],
            lw=3,
            label=f"column {k}",
        )
    # axes[2].legend()

    axes[0].set_title(title)
    axes[-1].set(xlabel="times (s)")


# ESTIMATE THE FRACTION OF THE EFFECT OF ux, uy and theta IN THE VALUE OF THE DELTA FUCTION INTEGRATED OVER TIME


def fraction_uk_filds_contained_in_delta_objective_value(
    solutionData_init_guess_cg: SolutionData,
    solutionData_op_cg: SolutionData,
    solutionData_mg: SolutionData,
    optimization: OptimizationProblem,
    integration_delay: bool = True,
):
    # retrieve the objective values
    ob_value_initial_guess_cg = optimization.objective_values[0]
    ob_value_optimised_cg = optimization.objective_values[-1]
    if integration_delay == True:
        t_min = 200 * optimization.forward_problem.n_cycle_min
    # estimate the fraction of uk fields of the op cg
    ux_op_cg = (
        (
            (
                solutionData_op_cg.fields[
                    t_min:,
                    0,
                    optimization.forward_problem.cloaked_geometry.cgIDs_surronding_area,
                    0,
                ]
                - solutionData_mg.fields[
                    t_min:,
                    0,
                    optimization.forward_problem.cloaked_geometry.mgIDs_surronding_area,
                    0,
                ]
            )
            / optimization.forward_problem.spacing
        )
        ** 2
    ).sum() / (
        jnp.max(
            (
                (
                    solutionData_mg.fields[
                        t_min:,
                        0,
                        optimization.forward_problem.cloaked_geometry.mgIDs_surronding_area,
                        :,
                    ]
                    * optimization.dimension_less
                )
                ** 2
            ).sum(axis=(1, 2))
        )
        * ob_value_optimised_cg**2
    )
    uy_op_cg = (
        (
            (
                solutionData_op_cg.fields[
                    t_min:,
                    0,
                    optimization.forward_problem.cloaked_geometry.cgIDs_surronding_area,
                    1,
                ]
                - solutionData_mg.fields[
                    t_min:,
                    0,
                    optimization.forward_problem.cloaked_geometry.mgIDs_surronding_area,
                    1,
                ]
            )
            / optimization.forward_problem.spacing
        )
        ** 2
    ).sum() / (
        jnp.max(
            (
                (
                    solutionData_mg.fields[
                        t_min:,
                        0,
                        optimization.forward_problem.cloaked_geometry.mgIDs_surronding_area,
                        :,
                    ]
                    * optimization.dimension_less
                )
                ** 2
            ).sum(axis=(1, 2))
        )
        * ob_value_optimised_cg**2
    )
    utheta_op_cg = (
        (
            solutionData_op_cg.fields[
                t_min:,
                0,
                optimization.forward_problem.cloaked_geometry.cgIDs_surronding_area,
                2,
            ]
            - solutionData_mg.fields[
                t_min:,
                0,
                optimization.forward_problem.cloaked_geometry.mgIDs_surronding_area,
                2,
            ]
        )
        ** 2
    ).sum() / (
        jnp.max(
            (
                (
                    solutionData_mg.fields[
                        t_min:,
                        0,
                        optimization.forward_problem.cloaked_geometry.mgIDs_surronding_area,
                        :,
                    ]
                    * optimization.dimension_less
                )
                ** 2
            ).sum(axis=(1, 2))
        )
        * ob_value_optimised_cg**2
    )

    # estimate the fraction of uk fields of the initial guess cg
    ux_ig_cg = (
        (
            (
                solutionData_init_guess_cg.fields[
                    t_min:,
                    0,
                    optimization.forward_problem.cloaked_geometry.cgIDs_surronding_area,
                    0,
                ]
                - solutionData_mg.fields[
                    t_min:,
                    0,
                    optimization.forward_problem.cloaked_geometry.mgIDs_surronding_area,
                    0,
                ]
            )
            / optimization.forward_problem.spacing
        )
        ** 2
    ).sum() / (
        jnp.max(
            (
                (
                    solutionData_mg.fields[
                        t_min:,
                        0,
                        optimization.forward_problem.cloaked_geometry.mgIDs_surronding_area,
                        :,
                    ]
                    * optimization.dimension_less
                )
                ** 2
            ).sum(axis=(1, 2))
        )
        * ob_value_initial_guess_cg**2
    )
    uy_ig_cg = (
        (
            (
                solutionData_init_guess_cg.fields[
                    t_min:,
                    0,
                    optimization.forward_problem.cloaked_geometry.cgIDs_surronding_area,
                    1,
                ]
                - solutionData_mg.fields[
                    t_min:,
                    0,
                    optimization.forward_problem.cloaked_geometry.mgIDs_surronding_area,
                    1,
                ]
            )
            / optimization.forward_problem.spacing
        )
        ** 2
    ).sum() / (
        jnp.max(
            (
                (
                    solutionData_mg.fields[
                        t_min:,
                        0,
                        optimization.forward_problem.cloaked_geometry.mgIDs_surronding_area,
                        :,
                    ]
                    * optimization.dimension_less
                )
                ** 2
            ).sum(axis=(1, 2))
        )
        * ob_value_initial_guess_cg**2
    )
    utheta_ig_cg = (
        (
            solutionData_init_guess_cg.fields[
                t_min:,
                0,
                optimization.forward_problem.cloaked_geometry.cgIDs_surronding_area,
                2,
            ]
            - solutionData_mg.fields[
                t_min:,
                0,
                optimization.forward_problem.cloaked_geometry.mgIDs_surronding_area,
                2,
            ]
        )
        ** 2
    ).sum() / (
        jnp.max(
            (
                (
                    solutionData_mg.fields[
                        t_min:,
                        0,
                        optimization.forward_problem.cloaked_geometry.mgIDs_surronding_area,
                        :,
                    ]
                    * optimization.dimension_less
                )
                ** 2
            ).sum(axis=(1, 2))
        )
        * ob_value_initial_guess_cg**2
    )

    # plot
    species = ("optimized cloaked geometry", "initial guess")
    penguin_means = {
        "ux": (ux_op_cg, ux_ig_cg),
        "uy": (uy_op_cg, uy_ig_cg),
        "utheta": (utheta_op_cg, utheta_ig_cg),
    }

    x = np.arange(len(species))  # the label locations
    width = 0.25  # the width of the bars
    multiplier = 0

    fig, ax = plt.subplots(layout="constrained")

    for attribute, measurement in penguin_means.items():
        offset = width * multiplier
        rects = ax.bar(x + offset, measurement, width, label=attribute)
        ax.bar_label(rects, padding=3)
        multiplier += 1

    # Add some text for labels, title and custom x-axis tick labels, etc.
    ax.set_ylabel("fraction (dim less)")
    ax.set_title("Fraction of ux, uy, utheta in the objective value")
    ax.set_xticks(x + width, species)
    ax.legend(loc="upper left")  # , ncols=3)
    ax.set_ylim(0, 1)
    plt.show()


def plot_sketch(cloak_geo):
    # Plots to visualize region of the cloak and void
    blocks_for_plot = cloak_geo.block_centroids()
    fig, axes = plt.subplots(constrained_layout=True)
    axes.plot(*jnp.array(cloak_geo.void).T)
    axes.scatter(
        blocks_for_plot[cloak_geo.cgIDs_cloak_area, 0],
        blocks_for_plot[cloak_geo.cgIDs_cloak_area, 1],
        c="red",
    )
    axes.scatter(
        blocks_for_plot[cloak_geo.cgIDs_surronding_area, 0],
        blocks_for_plot[cloak_geo.cgIDs_surronding_area, 1],
        c="green",
    )
    axes.axis("equal")
    return fig, axes

# Design a cloak area

In [ ]:
n1_cells = 15
n2_cells = 15
spacing = 15.0  # 15 mm
bond_length = 0.15 * spacing

mother_geometry = KagomeGeometry(
    n1_cells=n1_cells,
    n2_cells=n2_cells,
    bond_length=bond_length,
    direct_basis=spacing * jnp.eye(2),
)

mother_geometry.compute_geometry()

(
    block_centroids_mg,
    centroid_node_vectors_mg,
    bond_connectivity_mg,
    reference_bond_vectors_mg,
) = mother_geometry.get_parametrization()


initial_shifts_1_mg = jnp.zeros(
    (mother_geometry.n1_cells + 1, mother_geometry.n2_cells, 2)
)
initial_shifts_2_mg = jnp.zeros(
    (mother_geometry.n1_cells, mother_geometry.n2_cells + 1, 2)
)
initial_shifts_3_mg = jnp.zeros((mother_geometry.n1_cells, mother_geometry.n2_cells, 2))

# Perturb shift in a periodic way
bias = 0.1*spacing
initial_shifts_1_mg = initial_shifts_1_mg.at[::2, ::2, 1].set(bias)
initial_shifts_1_mg = initial_shifts_1_mg.at[1::2, 1::2, 1].set(-bias)
initial_shifts_2_mg = initial_shifts_2_mg.at[::2, ::2, 1].set(bias)
initial_shifts_2_mg = initial_shifts_2_mg.at[1::2, 1::2, 1].set(-bias)
initial_shifts_3_mg = initial_shifts_3_mg.at[::2, ::2, 1].set(bias)
initial_shifts_3_mg = initial_shifts_3_mg.at[1::2, 1::2, 1].set(-bias)

## Circle Void
width_strip_cloak_area = 60.0 / 1.5
R = 35.0
N = 50
x0 = 150.0 / 4 * 3
y0 = 150.0 / 4 * 3
void = [
    [R * np.cos(2 * np.pi * k / N) + x0, R * np.sin(2 * np.pi * k / N) + y0]
    for k in range(N + 1)
]

cloaked_geometry = CloakKagomeGeometry(
    mother_geometry, block_centroids_mg(initial_shifts_1_mg, initial_shifts_2_mg, initial_shifts_3_mg), void, width_strip_cloak_area
)

cloaked_geometry.compute_geometry()

plot_sketch(cloaked_geometry)

In [ ]:
plt.close('all')
plot_geometry(
    cloaked_geometry.block_centroids(),
    cloaked_geometry.centroid_node_vectors(),
    cloaked_geometry.bond_connectivity(),
)

In [ ]:
# Initial guess
guessed_shifts_1, guessed_shifts_2, guessed_shifts_3 = (
    initial_shifts_1_mg[cloaked_geometry.mask_shifts_1],
    initial_shifts_2_mg[cloaked_geometry.mask_shifts_2],
    initial_shifts_3_mg[cloaked_geometry.mask_shifts_3],
)

# Mechanical params
k_stretch = 120.0  # stretching stiffness 120. N/mm
k_shear = 1.19  # shearing stiffness 1.19 N/mm
k_rot = 1.50  # rotational stiffness 1.50 Nmm
density = (
    1.0  # 6.18e-9  # Mg/mm^2 # NOTE: This is scaled to 1. just for static problems
)
damping_scaling = 1.0
damping = (
    0.0186
    * jnp.array(
        [
            2 * (0.36125 * density * spacing**2 * k_shear) ** 0.5,
            2 * (0.36125 * density * spacing**2 * k_shear) ** 0.5,
            2 * (0.02175026 * density * spacing**4 * k_rot) ** 0.5,
        ]
    )
    * damping_scaling
)


# Forward input for the two problems to be optimized
amplitude = 1.0 * spacing
n_timepoints = 200
simulation_time = 7000.0
optimization_name = "kagome_static_cloaking_3dp_pla_shims"
# Hz loading frequency for dynamic input
forward_input = ForwardInput(
    # amplitude=amplitude,  # mm
    # loading_rate=loading_rate,  # Hz
    shifts_1=guessed_shifts_1,
    shifts_2=guessed_shifts_2,
    shifts_3=guessed_shifts_3,
)

# Forward problem
problem = ForwardProblem(
    n1_cells=n1_cells,
    n2_cells=n2_cells,
    bond_length=bond_length,
    spacing=spacing,
    void=void,
    width_strip_cloak_area=width_strip_cloak_area,
    shifts_1_2_3_mg=(initial_shifts_1_mg, initial_shifts_2_mg, initial_shifts_3_mg),
    k_stretch=k_stretch,
    k_shear=k_shear,
    k_rot=k_rot,
    density=density,
    damping=damping,
    k_contact=k_rot,
    min_angle=-15 * jnp.pi / 180,
    cutoff_angle=-10 * jnp.pi / 180,
    amplitude=amplitude,  # mm
    simulation_time=simulation_time,
    n_timepoints=n_timepoints,
    name=optimization_name,
    # atol=1e-4,
)

optimization = OptimizationProblem(
    forward_problem=problem,
    forward_input=forward_input,
    objective_type="integrated",
    name=optimization_name,
)
problem_filename_prefix = f"kagome{'_linearized_strains' if optimization.forward_problem.linearized_strains else ''}_{optimization.forward_problem.n1_cells}x{optimization.forward_problem.n2_cells}_amplitude_{optimization.forward_problem.amplitude}"
optimization_filename = f"opt_{optimization.objective_type}_with_angle_30_and_length_3_constraints_{problem_filename_prefix}_void_circle_bias_{bias/spacing:.2f}"


# Setup forward problem
problem.setup()

### Intact geometry


In [ ]:
plt.close('all')
xlim, ylim = mother_geometry.get_xy_limits(
    initial_shifts_1_mg, initial_shifts_2_mg, initial_shifts_3_mg
) + 0.5 * spacing * jnp.array([-1, 1])

solutionData_mg = problem.solutionData_mg
generate_animation(
    solutionData_mg,
    field="u",
    deformed=True,
    out_filename=f"../out/{optimization.name}/{optimization_filename}/{problem_filename_prefix}_intact",
    xlim=xlim,
    ylim=ylim,
    cmap="inferno",
    fps=30,
    dpi=300,
    frame_range=range(0, len(solutionData_mg.timepoints), 2),
    figsize=(6, 5),
)

### Holed geometry


In [ ]:
xlim, ylim = mother_geometry.get_xy_limits(
    initial_shifts_1_mg, initial_shifts_2_mg, initial_shifts_3_mg
) + 0.5 * spacing * jnp.array([-1, 1])

design_value = guessed_shifts_1, guessed_shifts_2, guessed_shifts_3
solution_data_cg_initial_guess = problem.solve(design_value)
generate_animation(
    solution_data_cg_initial_guess,
    field="u",
    deformed=True,
    out_filename=f"../out/{optimization.name}/{optimization_filename}/{problem_filename_prefix}_holed",
    xlim=xlim,
    ylim=ylim,
    cmap="inferno",
    fps=30,
    dpi=300,
    frame_range=range(0, len(solution_data_cg_initial_guess.timepoints), 2),
    figsize=(6, 5),
)

### Disturbance (error: holed - intact)

In [ ]:
problem.cloaked_geometry.spacing = spacing

generate_delta_difference(
    solution_data_cg_initial_guess,
    problem.solutionData_mg,
    problem.cloaked_geometry,
    title="Disturbance (holed - intact)",
)


# Optimization of mechanical cloak

### Import most recent optimization object

In [ ]:
optimization = OptimizationProblem.from_dict(
    load_data(
        f"../data/{optimization.name}/{optimization_filename}.pkl",
    )
)

### Run optimization

In [ ]:
# optimization.run_optimization_nlopt(
#     initial_guess=(
#         optimization.forward_input.shifts_1,
#         optimization.forward_input.shifts_2,
#         optimization.forward_input.shifts_3,
#     ),
#     # initial_guess=optimization.design_values[-1],
#     n_iterations=60,
#     min_block_angle=10 * jnp.pi / 180,
#     min_void_angle=0 * jnp.pi / 180,
#     min_edge_length=1.0,  # mm
#     max_time=12*60*60,  # 12 hours
# )

# save_data(
#     f"../data/{optimization.name}/{optimization_filename}.pkl",
#     optimization.to_dict(),  # Optimization problem
# )

## Plots

### Import most recent optimization object


In [ ]:
optimization = OptimizationProblem.from_dict(
    load_data(
        f"../data/{optimization.name}/{optimization_filename}.pkl",
    )
)

### Objective iterations

In [ ]:
plot_objective_iterations(
    optimization=optimization,
    # optimization_filename=optimization_filename
)

### Plot designs


In [ ]:
for solution_data, label in zip(
    optimization.forward_problem.solution_data,
    ["intact", "holed", "cloaked"]
):
    fig, axes = plot_geometry(
        block_centroids=solution_data.block_centroids,
        centroid_node_vectors=solution_data.centroid_node_vectors,
        bond_connectivity=solution_data.bond_connectivity,
        figsize=(4, 4),
    )
    axes.axis("off")
    fig.savefig(
        f"../out/{optimization.name}/{optimization_filename}/{label}_geometry.png",
        dpi=300,
        transparent=True,
    )
    plt.close(fig)


### Snapshots

In [ ]:
cmap = matplotlib.colors.LinearSegmentedColormap.from_list(
    name="custom_cmap",
    colors=[
        "#ff7b00",  # negative
        "#e46e00",  # negative
        "#c86100",  # negative
        "#b25600",  # negative
        "#000000",  # 0
        "#0098b0",  # positive
        "#00aac4",  # positive
        "#00c1df",  # positive
        "#00ddff",  # positive
    ],
)
plt.close("all")
fig, axes = plt.subplots(
    ncols=3, figsize=(3 * 3.75, 3.5), sharex=True, constrained_layout=True
)
vlim = (
    min(
        data.fields[:, 0, :, 1].min()
        for data in optimization.forward_problem.solution_data
    ),
    max(
        data.fields[:, 0, :, 1].max()
        for data in optimization.forward_problem.solution_data
    ),
)
vlim = -jnp.max(jnp.abs(jnp.array(vlim))), jnp.max(jnp.abs(jnp.array(vlim)))
vlim = (-8.2, 8.2)  # Custom to have same scale as previous plots for comparison

for solution_data, ax in zip(optimization.forward_problem.solution_data, axes):
    plot_geometry_field_overlaid(
        data=solution_data,
        field="uy",
        timepoint=-1,
        deformed=True,
        cmap=cmap,
        vlim=vlim,
        colorbar=False,
        axis=False,
        ax=ax,
    )
    ax.axis("off")
    ax.set_aspect("equal")

# Add colorbar to the last axis
cb = fig.colorbar(
    matplotlib.cm.ScalarMappable(
        cmap=cmap, norm=matplotlib.colors.Normalize(vmin=vlim[0], vmax=vlim[1])
    ),
    ax=axes[-1],
    orientation="vertical",
    pad=0.08,
    aspect=25,
)
cb.ax.tick_params(labelsize=14)
cb.set_label(r"Vertical displacement $u_y$ [mm]", fontsize=16)
fig.savefig(
    f"../out/{optimization.name}/{optimization_filename}/uy_displacement_reference_holed_cloaked.png",
    dpi=300,
    bbox_inches="tight",
)

### Response animation: Optimized cloaked geometry

In [ ]:
xlim, ylim = mother_geometry.get_xy_limits(
    initial_shifts_1_mg, initial_shifts_2_mg, initial_shifts_3_mg
) + 0.5 * spacing * jnp.array([-1, 1])

solution_data_cg_optimized_geometry = optimization.forward_problem.solution_data[-1]
generate_animation(
    solution_data_cg_optimized_geometry,
    field="u",
    deformed=True,
    out_filename=f"../out/{optimization.name}/{optimization_filename}/{problem_filename_prefix}_cloaked",
    xlim=xlim,
    ylim=ylim,
    cmap="inferno",
    fps=30,
    dpi=300,
    frame_range=range(0, len(solution_data_cg_optimized_geometry.timepoints), 2),
    figsize=(6, 5),
)


### Response animation: Intact vs holed vs cloaked geometry


In [ ]:
cmap = matplotlib.colors.LinearSegmentedColormap.from_list(
    name="custom_cmap",
    colors=[
        "#ff7b00",  # negative
        "#e46e00",  # negative
        "#c86100",  # negative
        "#b25600",  # negative
        "#000000",  # 0
        "#0098b0",  # positive
        "#00aac4",  # positive
        "#00c1df",  # positive
        "#00ddff",  # positive
    ],
)
vlim = (
    min(
        data.fields[:, 0, :, 1].min()
        for data in optimization.forward_problem.solution_data
    ),
    max(
        data.fields[:, 0, :, 1].max()
        for data in optimization.forward_problem.solution_data
    ),
)
vlim = -jnp.max(jnp.abs(jnp.array(vlim))), jnp.max(jnp.abs(jnp.array(vlim)))

xlim, ylim = mother_geometry.get_xy_limits(
    initial_shifts_1_mg, initial_shifts_2_mg, initial_shifts_3_mg
) + 0.5 * spacing * jnp.array([-1, 1])

plt.close("all")
generate_several_animations(
    # [solutionData_mg, solution_data_cg_initial_guess, solution_data_cg_optimized_geometry]
    optimization.forward_problem.solution_data,
    field="uy",
    out_filename="gigi",#f"../out/{optimization.name}/{optimization_filename}/{problem_filename_prefix}_intact_holed_cloaked_comparison_uy_custom_cmap",
    row_or_column="row",
    xlim=xlim,
    ylim=ylim,
    fps=30,
    dpi=300,
    cmap=cmap,
    vlim=vlim,
    frame_range=range(
        0, len(optimization.forward_problem.solution_data[0].timepoints), 2
    ),
    figsize=(12.5, 4),
    # legend_label="Displacement [mm]",
    legend_label="Displacement $u_y$ [mm]",
    axis=False,
)

# Perturbed reference design

In [ ]:
n1_cells = 15
n2_cells = 15
spacing = 15.0  # 15 mm
bond_length = 0.15 * spacing

mother_geometry = KagomeGeometry(
    n1_cells=n1_cells,
    n2_cells=n2_cells,
    bond_length=bond_length,
    direct_basis=spacing * jnp.eye(2),
)

mother_geometry.compute_geometry()

(
    block_centroids_mg,
    centroid_node_vectors_mg,
    bond_connectivity_mg,
    reference_bond_vectors_mg,
) = mother_geometry.get_parametrization()


initial_shifts_1_mg = jnp.zeros(
    (mother_geometry.n1_cells + 1, mother_geometry.n2_cells, 2)
)
initial_shifts_2_mg = jnp.zeros(
    (mother_geometry.n1_cells, mother_geometry.n2_cells + 1, 2)
)
initial_shifts_3_mg = jnp.zeros((mother_geometry.n1_cells, mother_geometry.n2_cells, 2))

# Perturb shift in a periodic way
bias = 0.1*spacing
initial_shifts_1_mg = initial_shifts_1_mg.at[::2, ::2, 1].set(bias)
initial_shifts_1_mg = initial_shifts_1_mg.at[1::2, 1::2, 1].set(-bias)
initial_shifts_2_mg = initial_shifts_2_mg.at[::2, ::2, 1].set(bias)
initial_shifts_2_mg = initial_shifts_2_mg.at[1::2, 1::2, 1].set(-bias)
initial_shifts_3_mg = initial_shifts_3_mg.at[::2, ::2, 1].set(bias)
initial_shifts_3_mg = initial_shifts_3_mg.at[1::2, 1::2, 1].set(-bias)

# Perturb shift with some random noise
noise = 0.01 * spacing
key = 3
_key = jax.random.PRNGKey(key)
initial_shifts_1_mg = initial_shifts_1_mg + noise * jax.random.uniform(
    _key, initial_shifts_1_mg.shape, minval=-1.0, maxval=1.0
)
_key, subkey = jax.random.split(_key)
initial_shifts_2_mg = initial_shifts_2_mg + noise * jax.random.uniform(
    subkey, initial_shifts_2_mg.shape, minval=-1.0, maxval=1.0
)
_key, subkey = jax.random.split(_key)
initial_shifts_3_mg = initial_shifts_3_mg + noise * jax.random.uniform(
    subkey, initial_shifts_3_mg.shape, minval=-1.0, maxval=1.0
)

## Circle Void
width_strip_cloak_area = 60.0 / 1.5
R = 35.0
N = 50
x0 = 150.0 / 4 * 3
y0 = 150.0 / 4 * 3
void = [
    [R * np.cos(2 * np.pi * k / N) + x0, R * np.sin(2 * np.pi * k / N) + y0]
    for k in range(N + 1)
]


cloaked_geometry = CloakKagomeGeometry(
    mother_geometry, block_centroids_mg(initial_shifts_1_mg, initial_shifts_2_mg, initial_shifts_3_mg), void, width_strip_cloak_area
)

cloaked_geometry.compute_geometry()

# plot_sketch(cloaked_geometry)

In [ ]:
plt.close('all')
plot_geometry(
    mother_geometry.block_centroids(initial_shifts_1_mg, initial_shifts_2_mg, initial_shifts_3_mg),
    mother_geometry.centroid_node_vectors(initial_shifts_1_mg, initial_shifts_2_mg, initial_shifts_3_mg),
    mother_geometry.bond_connectivity(),
    figsize=(4, 4)
)

In [ ]:
# Initial guess
guessed_shifts_1, guessed_shifts_2, guessed_shifts_3 = (
    initial_shifts_1_mg[cloaked_geometry.mask_shifts_1],
    initial_shifts_2_mg[cloaked_geometry.mask_shifts_2],
    initial_shifts_3_mg[cloaked_geometry.mask_shifts_3],
)

# Mechanical params
k_stretch = 120.0  # stretching stiffness 120. N/mm
k_shear = 1.19  # shearing stiffness 1.19 N/mm
k_rot = 1.50  # rotational stiffness 1.50 Nmm
density = (
    1.0  # 6.18e-9  # Mg/mm^2 # NOTE: This is scaled to 1. just for static problems
)
damping_scaling = 1.0
damping = (
    0.0186
    * jnp.array(
        [
            2 * (0.36125 * density * spacing**2 * k_shear) ** 0.5,
            2 * (0.36125 * density * spacing**2 * k_shear) ** 0.5,
            2 * (0.02175026 * density * spacing**4 * k_rot) ** 0.5,
        ]
    )
    * damping_scaling
)

# Forward input for the two problems to be optimized
amplitude = 1.0 * spacing
n_timepoints = 200
simulation_time = 7000.0
optimization_name = "kagome_static_cloaking_3dp_pla_shims"
# Hz loading frequency for dynamic input
forward_input = ForwardInput(
    # amplitude=amplitude,  # mm
    # loading_rate=loading_rate,  # Hz
    shifts_1=guessed_shifts_1,
    shifts_2=guessed_shifts_2,
    shifts_3=guessed_shifts_3,
)

# Forward problem
problem = ForwardProblem(
    n1_cells=n1_cells,
    n2_cells=n2_cells,
    bond_length=bond_length,
    spacing=spacing,
    void=void,
    width_strip_cloak_area=width_strip_cloak_area,
    shifts_1_2_3_mg=(initial_shifts_1_mg, initial_shifts_2_mg, initial_shifts_3_mg),
    k_stretch=k_stretch,
    k_shear=k_shear,
    k_rot=k_rot,
    density=density,
    damping=damping,
    k_contact=k_rot,
    min_angle=-15 * jnp.pi / 180,
    cutoff_angle=-10 * jnp.pi / 180,
    amplitude=amplitude,  # mm
    simulation_time=simulation_time,
    n_timepoints=n_timepoints,
    name=optimization_name,
    # atol=1e-4,
)

optimization = OptimizationProblem(
    forward_problem=problem,
    forward_input=forward_input,
    objective_type="integrated",
    name=optimization_name,
)
problem_filename_prefix = f"kagome{'_linearized_strains' if optimization.forward_problem.linearized_strains else ''}_{optimization.forward_problem.n1_cells}x{optimization.forward_problem.n2_cells}_amplitude_{optimization.forward_problem.amplitude}"
optimization_filename = f"opt_{optimization.objective_type}_with_angle_30_and_length_3_constraints_{problem_filename_prefix}_void_circle_bias_{bias/spacing:.2f}"


# Setup forward problem
problem.setup()

### Plot reference design


In [ ]:
for solution_data, label in zip(
    [problem.solutionData_mg], ["intact"]
):
    fig, axes = plot_geometry(
        block_centroids=solution_data.block_centroids,
        centroid_node_vectors=solution_data.centroid_node_vectors,
        bond_connectivity=solution_data.bond_connectivity,
        figsize=(4, 4),
    )
    axes.axis("off")
    fig.savefig(
        f"../out/{optimization.name}/{optimization_filename}/{label}_geometry_noise_{noise/spacing:.2f}.png",
        dpi=300,
        transparent=True,
    )
    plt.close(fig)


### Snapshots

In [ ]:
cmap = matplotlib.colors.LinearSegmentedColormap.from_list(
    name="custom_cmap",
    colors=[
        "#ff7b00",  # negative
        "#e46e00",  # negative
        "#c86100",  # negative
        "#b25600",  # negative
        "#000000",  # 0
        "#0098b0",  # positive
        "#00aac4",  # positive
        "#00c1df",  # positive
        "#00ddff",  # positive
    ],
)
plt.close("all")
fig, axes = plt.subplots(
    figsize=(4, 3), sharex=True, constrained_layout=True
)
vlim = (
    min(
        data.fields[:, 0, :, 1].min()
        for data in [problem.solutionData_mg]
    ),
    max(
        data.fields[:, 0, :, 1].max()
        for data in [problem.solutionData_mg]
    ),
)
vlim = -jnp.max(jnp.abs(jnp.array(vlim))), jnp.max(jnp.abs(jnp.array(vlim)))
vlim = (-8.2, 8.2)  # Custom to have same scale as previous plots for comparison

plot_geometry_field_overlaid(
    data=problem.solutionData_mg,
    field="uy",
    timepoint=-1,
    deformed=True,
    cmap=cmap,
    vlim=vlim,
    colorbar=False,
    axis=False,
    ax=axes,
)
axes.axis("off")
axes.set_aspect("equal")

# Add colorbar to the last axis
cb = fig.colorbar(
    matplotlib.cm.ScalarMappable(
        cmap=cmap, norm=matplotlib.colors.Normalize(vmin=vlim[0], vmax=vlim[1])
    ),
    ax=axes,
    orientation="vertical",
    pad=0.1,
    aspect=30,
)
cb.ax.tick_params(labelsize=14)
cb.set_label(r"$u_y$ [mm]", fontsize=16)
fig.savefig(
    f"../out/{optimization.name}/{optimization_filename}/uy_displacement_reference_noise_{noise/spacing:.2f}.png",
    dpi=300,
    bbox_inches="tight",
)

In [ ]:
fig, axes = plt.subplots(
    figsize=(4, 5), sharex=True, constrained_layout=True
)
vlim = (-8.2, 8.2)  # Custom to have same scale as previous plots for comparison
cb = fig.colorbar(
    matplotlib.cm.ScalarMappable(
        cmap=cmap, norm=matplotlib.colors.Normalize(vmin=vlim[0], vmax=vlim[1])
    ),
    ax=axes,
    orientation="vertical",
    pad=0.1,
    aspect=35,
)
cb.ax.tick_params(labelsize=14)
cb.set_label(r"Vertical displacement $u_y$ [mm]", fontsize=16)
# Save colorbar only
fig.savefig(
    f"../out/{optimization.name}/{optimization_filename}/uy_displacement_colorbar_only_noise_{noise/spacing:.2f}.png",
    dpi=300,
    bbox_inches="tight",
)